# Визуализация диалогов: Мульти-модельная оценка

25-шаговый клинический encounter (PMC10783203) — врач vs ИИ-пациент

## Промпт
```yaml
SYSTEM_PROMPT = """\
Ты играешь роль пациента в симуляции медицинского образования. Ты не врач.
ЯЗЫК: Русский. Отвечай только на русском языке.

ПРАВИЛА:
- Используй только факты, перечисленные ниже (закрытый мир). Если спрашивают
  о чём-то не перечисленном — скажи, что не знаешь, не помнишь или не замечал.
- Не называй медицинский диагноз и не используй медицинские термины
  (сердечный приступ, перикардит, миокардит и т.п.).
- Прогрессивное раскрытие: на общие вопросы («расскажите о себе», «что вас
  беспокоит», «как я могу помочь») используй ТОЛЬКО блок ПЕРВЫЙ ВИЗИТ.
  Подробности об иррадиации, что усиливает/облегчает боль, недавней простуде,
  вредных привычках и семейном анамнезе — только когда врач спрашивает.
- Ответы короткие (1–3 предложения), если не попросят подробнее.
- Тон: тревожный — ты напуган, но стараешься чётко отвечать;
  не говори длинными монологами.

КТО ТЫ:
- 24 года, мужчина.
- Студент университета, раньше был здоров, напуган болью но старается отвечать.

АНАМНЕЗ ЖИЗНИ (сообщай только если спрашивают):
- Хронических заболеваний нет; постоянных лекарств не принимает; аллергий не знает.
- Курение: никогда.
- Алкоголь: несколько раз в неделю, немного.
- Работа/учёба: студент.
- Семья: о ранних инфарктах или внезапных смертях у молодых родственников не слышал.

--- ПЕРВЫЙ ВИЗИТ / ТОЛЬКО ОБЩИЕ ВОПРОСЫ ---
Можешь сказать, что тебе 24, ты студент, и у тебя очень сильная боль
в середине груди, которая началась сегодня пока спал — точно не знаешь
сколько часов назад. Это самая сильная боль в груди в твоей жизни, и ты
напуган.
Стоп. Не упоминай обе руки, колющий характер, лучше когда сидишь,
недавнюю простуду или отсутствие тошноты — до тех пор пока не спросят.

--- ПОЛНЫЙ АНАМНЕЗ (раскрывай постепенно по вопросам) ---
Самая сильная боль в груди в твоей жизни. Началась пока спал.
Отдаёт в обе руки. Иногда колющая. Хуже если ходить или лежать;
немного лучше если сидеть или наклониться вперёд.
Немного не хватает дыхания. Примерно полторы недели назад была простуда.

Структурированные факты (не выдавай все сразу):
- Начало: во сне, в день обращения.
- Длительность: примерно шесть часов к моменту обращения (можно приблизительно).
- Локализация: за грудиной.
- Иррадиация: обе руки.
- Характер: очень сильная, иногда колющая.
- Интенсивность: самая сильная боль в жизни.
- Сопутствующее: лёгкая одышка.

Негативы — на прямой вопрос ВСЕГДА отрицай:
- Тошноты не было.
- Сильного пота не было (может быть слегка влажным от страха — не «в поту»).
- Наркотики не употребляет.
- Никуда не выезжал в последнее время.
- В этом году прививок не делал (насколько помнит).

ЧТО ТЫ ЕЩЁ НЕ ЗНАЕШЬ:
- Результаты анализов крови, тропонин, ЭКГ, рентген, ЭХО, МРТ — до тех пор
  пока врач тебе не сообщит."""

```

In [6]:
import json, glob, os
from IPython.display import HTML, display

result_files = sorted(glob.glob("results_*.json"))
print(f"Найдено файлов: {len(result_files)}")
for f in result_files:
    
    print(f"  {f}")

Найдено файлов: 8
  results_Claude_Sonnet_4.6.json
  results_Claude_Sonnet_4.6_c2.json
  results_Claude_Sonnet_4.6_t0_7.json
  results_Claude_Sonnet_4.6_t0_7_c2.json
  results_Ministral_8B_OpenRouter.json
  results_Ministral_8B_OpenRouter_c2.json
  results_Qwen_2.5_72B_OpenRouter.json
  results_Qwen_2.5_72B_OpenRouter_c2.json


In [7]:
def render_dialogue(run_data, model_name, temperature=0.3, case_num=1, case_info=None):
    steps = run_data["steps"]
    n_passed = run_data["n_passed"]
    n_total = run_data["n_total"]
    first_fail = run_data["first_fail_step"]
    usage = run_data["usage"]
    run_num = run_data["run"]

    case_label = f"Кейс #{case_num}"
    if case_info:
        case_label += f": {case_info.get('description', '')}"

    html = []
    html.append('<div style="color-scheme:light;color:#111827;max-width:900px;margin:20px auto;font-family:system-ui,sans-serif;">')

    # Header
    html.append(f'<div style="background:#1a1a2e;color:#eee;padding:16px 24px;border-radius:12px 12px 0 0;">')
    html.append(f'<h2 style="margin:0 0 8px 0;font-size:18px;">{model_name} — Прогон {run_num} (t={temperature}, {case_label})</h2>')
    html.append(f'<span style="font-size:13px;color:#e2e8f0;">Результат: {n_passed}/{n_total} прошли | Первый сбой: шаг {first_fail} | Токены: in={usage["input_tokens"]}, out={usage["output_tokens"]}</span>')
    html.append(f'</div>')

    for s in steps:
        step_num = s["step"]
        phase = s["phase"]
        cls = s["class"]
        doctor = s["doctor"]
        patient = s["patient_response"]
        passed = s["passed"]
        fail_reason = s["fail_reason"]

        # Step header
        bg = "#c8e6c9" if passed else "#ffcdd2"
        border = "#4caf50" if passed else "#f44336"
        badge = "PASS" if passed else f"FAIL: {fail_reason}"
        badge_bg = "#4caf50" if passed else "#f44336"

        html.append(f'<div style="border-left:4px solid {border};background:{bg};margin:0;padding:12px 20px;">')
        html.append(f'<div style="display:flex;align-items:center;gap:12px;margin-bottom:8px;">')
        html.append(f'<span style="font-weight:700;font-size:14px;color:#111827;">Шаг {step_num}</span>')
        html.append(f'<span style="font-size:12px;color:#374151;">[{phase}]</span>')
        html.append(f'<span style="font-size:11px;background:#e0e0e0;color:#111827;padding:2px 8px;border-radius:10px;font-weight:600;">{cls}</span>')
        html.append(f'<span style="font-size:11px;background:{badge_bg};color:#fff;padding:2px 10px;border-radius:10px;font-weight:600;">{badge}</span>')
        html.append(f'</div>')

        # Doctor
        html.append(f'<div style="margin-bottom:8px;">')
        html.append(f'<span style="font-size:12px;font-weight:600;color:#1565c0;">Врач:</span> ')
        html.append(f'<span style="font-size:13px;color:#111827;">{doctor}</span>')
        html.append(f'</div>')

        # Patient
        html.append(f'<div style="background:#fff;border-radius:8px;padding:10px 14px;margin-left:20px;border:1px solid #9e9e9e;">')
        html.append(f'<span style="font-size:12px;font-weight:600;color:#e65100;">Пациент:</span> ')
        html.append(f'<span style="font-size:13px;color:#111827;">{patient}</span>')
        html.append(f'</div>')

        html.append(f'</div>')

    # Footer
    html.append(f'<div style="background:#1a1a2e;color:#e2e8f0;padding:12px 24px;border-radius:0 0 12px 12px;text-align:center;font-size:12px;">')
    html.append(f'Конец диалога — {model_name}, Прогон {run_num}')
    html.append(f'</div>')
    html.append(f'</div>')

    return "\n".join(html)


def render_comparison_table():
    """Сводная таблица всех моделей."""
    html = []
    html.append('<div style="color-scheme:light;color:#111827;max-width:900px;margin:20px auto;font-family:system-ui,sans-serif;">')
    html.append('<h2 style="margin:0 0 16px 0;font-size:18px;color:#111827;">Сравнительная таблица</h2>')
    html.append('<table style="width:100%;border-collapse:collapse;font-size:13px;border:1px solid #757575;">')
    html.append('<tr style="background:#1a1a2e;color:#fff;">')
    for th in ["Модель", "Кейс", "Temp", "Прогон", "Прошли", "Первый сбой", "Input", "Output", "Стоимость"]:
        html.append(f'<th style="padding:10px 12px;text-align:left;border-bottom:2px solid #333;">{th}</th>')
    html.append('</tr>')

    pricing_map = {
        "Claude Sonnet 4.6": {"input": 3.0, "output": 15.0},
        "Qwen 2.5 72B (OpenRouter)": {"input": 0.65, "output": 0.65},
        "Ministral 8B (OpenRouter)": {"input": 0.1, "output": 0.1},
    }

    for fpath in sorted(glob.glob("results_*.json")):
        with open(fpath, encoding="utf-8") as f:
            data = json.load(f)
        model_name = data["model"]
        temperature = data.get("temperature", 0.3)
        runs = data.get("results", [])

        # Extract case info from filename
        fname = os.path.basename(fpath)
        case_num = 1
        if "_c2" in fname:
            case_num = 2

        for run in runs:
            if "error" in run:
                html.append(f'<tr style="color:#111827;background:#ffcdd2;border-bottom:1px solid #757575;"><td style="padding:8px 12px;">{model_name}</td><td style="padding:8px 12px;">#{case_num}</td><td style="padding:8px 12px;">{temperature}</td><td style="padding:8px 12px;">{run["run"]}</td><td colspan="5" style="padding:8px 12px;color:#b71c1c;font-weight:600;">ОШИБКА: {run["error"][:80]}</td></tr>')
                continue
            usage = run["usage"]
            pricing = pricing_map.get(model_name, {"input": 0, "output": 0})
            cost = (usage["input_tokens"] * pricing["input"] + usage["output_tokens"] * pricing["output"]) / 1_000_000
            _full = run["n_passed"] == 25
            bg = "#c8e6c9" if _full else "#ffe0b2"
            _accent = "#2e7d32" if _full else "#e65100"
            html.append(f'<tr style="color:#111827;background:{bg};border-bottom:1px solid #9e9e9e;border-left:4px solid {_accent};">')
            html.append(f'<td style="padding:8px 12px;color:#111827;">{model_name}</td>')
            html.append(f'<td style="padding:8px 12px;color:#111827;">#{case_num}</td>')
            html.append(f'<td style="padding:8px 12px;color:#111827;">{temperature}</td>')
            html.append(f'<td style="padding:8px 12px;color:#111827;">{run["run"]}</td>')
            html.append(f'<td style="padding:8px 12px;font-weight:700;color:#111827;">{run["n_passed"]}/25</td>')
            html.append(f'<td style="padding:8px 12px;color:#111827;">{run["first_fail_step"]}</td>')
            html.append(f'<td style="padding:8px 12px;color:#111827;">{usage["input_tokens"]}</td>')
            html.append(f'<td style="padding:8px 12px;color:#111827;">{usage["output_tokens"]}</td>')
            html.append(f'<td style="padding:8px 12px;color:#111827;">${cost:.4f}</td>')
            html.append('</tr>')

    html.append('</table>')
    html.append('</div>')
    return "\n".join(html)

## Сравнительная таблица

In [8]:
display(HTML(render_comparison_table()))

Модель,Кейс,Temp,Прогон,Прошли,Первый сбой,Input,Output,Стоимость
Claude Sonnet 4.6,#1,0.3,1,24/25,1,0,41341,$0.6201
Claude Sonnet 4.6,#1,0.3,2,23/25,1,0,35705,$0.5356
Claude Sonnet 4.6,#1,0.3,3,22/25,1,0,47235,$0.7085
Claude Sonnet 4.6,#2,0.3,1,22/25,1,0,38206,$0.5731
Claude Sonnet 4.6,#2,0.3,2,20/25,1,0,34761,$0.5214
Claude Sonnet 4.6,#2,0.3,3,23/25,1,0,33331,$0.5000
Claude Sonnet 4.6,#1,0.7,1,22/25,1,0,39979,$0.5997
Claude Sonnet 4.6,#1,0.7,2,24/25,1,0,41007,$0.6151
Claude Sonnet 4.6,#1,0.7,3,23/25,1,0,40100,$0.6015
Claude Sonnet 4.6,#2,0.7,1,21/25,1,0,44626,$0.6694


## Диалоги по моделям

Ниже выводятся **все прогоны** по **всем моделям** из `results_*.json` (успешные — полный HTML-чат; с ошибкой — строка в stdout). Между блоками — разделитель.

In [9]:
# Загружаем все результаты + кейс-конфиги для описаний
all_data = {}  # model_name -> { "data": ..., "case_num": ..., "temperature": ... }
for fpath in sorted(glob.glob("results_*.json")):
    with open(fpath, encoding="utf-8") as f:
        data = json.load(f)
    fname = os.path.basename(fpath)
    case_num = 2 if "_c2" in fname else 1
    temperature = data.get("temperature", 0.3)
    model_name = data["model"]
    key = f"{model_name} (t={temperature}, кейс #{case_num})"
    all_data[key] = {
        "data": data,
        "case_num": case_num,
        "temperature": temperature,
        "model_name": model_name,
    }

# Load case configs for descriptions
case_configs = {}
for cf in sorted(glob.glob("case_config*.json")):
    with open(cf, encoding="utf-8") as f:
        case_configs[cf] = json.load(f)

print("Доступные прогоны:")
for key, info in all_data.items():
    runs = info["data"].get("results", [])
    ok = [r for r in runs if "error" not in r]
    print(f"  {key}: {len(ok)} прогонов")

print("\nДоступные кейсы:")
for cf, cfg in case_configs.items():
    print(f"  {cf}: {cfg['case_info']['description']}")

Доступные прогоны:
  Claude Sonnet 4.6 (t=0.3, кейс #1): 3 прогонов
  Claude Sonnet 4.6 (t=0.3, кейс #2): 3 прогонов
  Claude Sonnet 4.6 (t=0.7, кейс #1): 3 прогонов
  Claude Sonnet 4.6 (t=0.7, кейс #2): 3 прогонов
  Ministral 8B (OpenRouter) (t=0.3, кейс #1): 3 прогонов
  Ministral 8B (OpenRouter) (t=0.3, кейс #2): 3 прогонов
  Qwen 2.5 72B (OpenRouter) (t=0.3, кейс #1): 3 прогонов
  Qwen 2.5 72B (OpenRouter) (t=0.3, кейс #2): 3 прогонов

Доступные кейсы:
  case_config.json: Young man with chest pain, high troponin — ACS vs myopericarditis
  case_config_2.json: Young male with sudden flank pain radiating to groin — renal colic vs lumbar radiculopathy


In [10]:
# Helper to get case info
def get_case_info(case_num):
    cf = f"case_config_{case_num}.json" if case_num != 1 else "case_config.json"
    return case_configs.get(cf, {}).get("case_info", None)

for key in sorted(all_data.keys()):
    info = all_data[key]
    model_name = info["model_name"]
    temperature = info["temperature"]
    case_num = info["case_num"]
    data = info["data"]
    runs = data.get("results", [])
    case_info = get_case_info(case_num)

    for run in sorted(runs, key=lambda r: r.get("run", 0)):
        rn = run.get("run", "?")
        if "error" in run:
            print(f"[{key}] прогон {rn}: ошибка — {run['error'][:200]}")
            continue
        display(HTML(render_dialogue(run, model_name, temperature=temperature, case_num=case_num, case_info=case_info)))
        display(HTML('<hr style="margin:28px 0;border:none;border-top:2px solid #9e9e9e"/>'))